In [ ]:
# %% Deep learning - Section 23.210
#    CNN GAN with Gaussians

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T
import torch.nn.utils         as utils
import random

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [68]:
# %% Generate data

n_images = 3000
img_size = 64

x   = np.linspace(-4,4,img_size)
X,Y = np.meshgrid(x,x)

# Preallocate images and labels tensors
images = torch.zeros(n_images,1,img_size,img_size)

for i in range(n_images):

    # Gaussian with random centre (ro is random offset)
    ro    = 2*np.random.randn(2)
    width = np.random.rand()/.6 + 1.8
    G     = np.exp( -( (X-ro[0])**2 + (Y-ro[1])**2) / (2*width**2) )

    # Add noise
    G  = G + np.random.randn(img_size,img_size)/5

    # Store to tensor
    images[i,:,:,:] = torch.Tensor(G).view(1,img_size,img_size)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5))/2
fig,axs = plt.subplots(3,7,figsize=(phi*7,7))

for i,ax in enumerate(axs.flatten()):

    pic = np.random.randint(n_images)
    G   = np.squeeze(images[pic,:,:])

    ax.imshow(G,vmin=-1,vmax=1,cmap='jet')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Some Gaussians')

plt.savefig('figure20_gan_cnn_gaussians.png')
plt.show()
files.download('figure20_gan_cnn_gaussians.png')


In [ ]:
# %% See also

# https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html


In [51]:
# %% Discriminator class

class DiscriminatorNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolution layers
        self.conv1 = nn.Conv2d(  1, 64, 4, 2, 1, bias=False)
        self.conv2 = nn.Conv2d( 64,128, 4, 2, 1, bias=False)
        self.conv3 = nn.Conv2d(128,256, 4, 2, 1, bias=False)
        self.conv4 = nn.Conv2d(256,512, 4, 2, 1, bias=False)
        self.conv5 = nn.Conv2d(512,  1, 4, 1, 0, bias=False)

        # Batchnorm
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(256)
        self.bn4 = nn.BatchNorm2d(512)

    def forward(self,x):

        x = F.leaky_relu( self.conv1(x) ,.2)

        x = F.leaky_relu( self.conv2(x) ,.2)
        x = self.bn2(x)

        x = F.leaky_relu( self.conv3(x) ,.2)
        x = self.bn3(x)

        x = F.leaky_relu( self.conv4(x) ,.2)
        x = self.bn4(x)

        x = torch.sigmoid( self.conv5(x) ).view(-1,1)

        return x


In [52]:
# %% Generator class

class GeneratorNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolution layers
        self.conv1 = nn.ConvTranspose2d(100,512, 4, 1, 0, bias=False)
        self.conv2 = nn.ConvTranspose2d(512,256, 4, 2, 1, bias=False)
        self.conv3 = nn.ConvTranspose2d(256,128, 4, 2, 1, bias=False)
        self.conv4 = nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False)
        self.conv5 = nn.ConvTranspose2d(64,   1, 4, 2, 1, bias=False)

        # Batchnorm
        self.bn1 = nn.BatchNorm2d(512)
        self.bn2 = nn.BatchNorm2d(256)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d( 64)

    def forward(self,x):

        x = F.relu( self.bn1(self.conv1(x)) )
        x = F.relu( self.bn2(self.conv2(x)) )
        x = F.relu( self.bn3(self.conv3(x)) )
        x = F.relu( self.bn4(self.conv4(x)) )
        x = torch.tanh( self.conv5(x) )

        return x


In [ ]:
# %% Test classes on random data

# Discriminator
d_net = DiscriminatorNet()
y     = d_net(torch.randn(10,1,64,64))
print(y.shape)
print(y), print()

# Generator
g_net = GeneratorNet()
y     = g_net(torch.randn(10,100,1,1))
print(y.shape)
print(y[0,:].detach().squeeze().view(64,64).shape)


In [81]:
# %% Set up the GAN ensemble

# Loss funtion is the same for both training phases
loss_fun = nn.BCELoss()

# Model instances
d_net = DiscriminatorNet().to(device)
g_net = GeneratorNet().to(device)

# Optimizer (same but two instances for the two models; GANs usually have a
# small lr and train slowly; betas are based on the tutorial listed above)
d_optimizer = torch.optim.Adam(d_net.parameters(), lr=.0002, betas=(.5,.999))
g_optimizer = torch.optim.Adam(g_net.parameters(), lr=.0002, betas=(.5,.999))


In [ ]:
# %% Train the GAN

# Takes ~4 mins on GPU with 1500 batches
num_epochs = 1500
batch_size = 86

# Preallocate
losses    = []
decisions = []

# Loop
for epoch_i in range(num_epochs):

    # Minibatch from randomly selected images
    r_idx = torch.randint(images.shape[0],(batch_size,))
    data  = images[r_idx,:].to(device)

    # Labels for real and fake images
    real_labels = torch.ones(batch_size,1).to(device)
    fake_labels = torch.zeros(batch_size,1).to(device)

    # Train the discriminator

    # Forward propagationa and loss for real images (all labels are 1)
    pred_real   = d_net(data)
    d_loss_real = loss_fun(pred_real,real_labels)

    # Forward propagation and loss for fake images (all lablels are 0)
    fake_data   = torch.randn(batch_size,100,1,1).to(device)
    fake_imgs   = g_net(fake_data)
    pred_fake   = d_net(fake_imgs)
    d_loss_fake = loss_fun(pred_fake,fake_labels)

    # Build combined loss (note as you can scale the loss to make the model more
    # or less sensitive to FP or FN)
    d_loss = 1*d_loss_real + 1*d_loss_fake

    # Backpropagation of discriminator
    d_optimizer.zero_grad()
    d_loss.backward()
    d_optimizer.step()

    # Train the generator

    # Create fake images and compute loss
    fake_imgs  = g_net( torch.randn(batch_size,100,1,1).to(device) )
    pred_fake  = d_net(fake_imgs)

    # Get loss and discrimination (pass real labels so that the model minimise
    # the loss between pred_fake and real_labels)
    g_loss = loss_fun(pred_fake,real_labels)

    # Backpropagation of generator
    g_optimizer.zero_grad()
    g_loss.backward()
    g_optimizer.step()

    # Collect losses and decisions
    losses.append([d_loss.item(),g_loss.item()])

    d1 = torch.mean((pred_real>.5).float()).detach().item()
    d2 = torch.mean((pred_fake>.5).float()).detach().item()
    decisions.append([d1,d2])

    # Status message
    if (epoch_i+1)%50==0:
        msg = f'Finished epoch {epoch_i+1}/{num_epochs}'
        sys.stdout.write('\r' + msg)

# Convert from list to numpy array
losses    = np.array(losses)
decisions = np.array(decisions)


In [62]:
# %% Functions for 1D smoothing filter

# Improved for edge effects - adaptive window
def smooth_adaptive(x,k):
    smoothed = np.zeros_like(x)
    half_k   = k // 2

    for i in range(len(x)):
        start       = max(0, i-half_k)
        end         = min(len(x), i+half_k + 1)
        smoothed[i] = np.mean(x[start:end])

    return smoothed


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,3,figsize=(2*phi*6,6))

ax[0].plot(smooth_adaptive(losses[:,0],50))
ax[0].plot(smooth_adaptive(losses[:,1],50))
ax[0].set_xlabel('Batches')
ax[0].set_ylabel('Loss')
ax[0].set_title('Model loss')
ax[0].legend(['Discrimator','Generator'])
# ax[0].set_xlim([500,900])
# ax[0].set_ylim([0,2.5])

ax[1].plot(losses[200:,0],losses[200:,1],'k.',alpha=.1)
ax[1].set_xlabel('Discriminator loss')
ax[1].set_ylabel('Generator loss')
ax[1].set_title('Disc. Loss vs. Gen. Loss')

ax[2].plot(smooth_adaptive(decisions[:,0],50))
ax[2].plot(smooth_adaptive(decisions[:,1],50))
ax[2].set_xlabel('Epochs')
ax[2].set_ylabel('Probablity ("real")')
ax[2].set_title('Discriminator output')
ax[2].legend(['Real','Fake'])

plt.savefig('figure21_gan_cnn_gaussians.png')
plt.show()
files.download('figure21_gan_cnn_gaussians.png')


In [ ]:
# %% Plotting

# Generate the images with the generator network
g_net.eval()
fake_data = g_net(torch.randn(batch_size,100,1,1).to(device)).cpu()

# Plot
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(3,6,figsize=(phi*6,6))

for i,ax in enumerate(axs.flatten()):
    ax.imshow(fake_data[i,:,].detach().squeeze(),cmap='jet')
    ax.axis('off')

plt.suptitle('Generated Gaussians')

plt.savefig('figure22_gan_cnn_gaussians.png')
plt.show()
files.download('figure22_gan_cnn_gaussians.png')


In [ ]:
# %% Exercise 1
#    Train the model for 200 epochs. Do the generated Gaussians still look like Gaussians?

# Not really, wide Gaussians become large blobs covering all the image, while
# smaller Gaussians resemble just some vague square shapes


In [ ]:
# %% Exercise 2
#    The images are set to 64x64. Does the model still work if you make the images larger, e.g., 91x91? How about if
#    they are smaller, e.g., 28x28 (the size of MNIST)?

# With 91x91 images it's pretty much the same, with 28x28 the GAN collapsed, no
# actual learning occurred and the genearated images stay pretty much noise, not
# sure if it's a general thing, but it kept happening over a couple of runs


In [57]:
# %% Discriminator class (91x91)

class DiscriminatorNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolution layers
        self.conv1 = nn.Conv2d(  1, 64, 4, 3, 0, bias=False)
        self.conv2 = nn.Conv2d( 64,128, 2, 2, 1, bias=False)
        self.conv3 = nn.Conv2d(128,256, 4, 2, 1, bias=False)
        self.conv4 = nn.Conv2d(256,512, 4, 2, 1, bias=False)
        self.conv5 = nn.Conv2d(512,  1, 4, 1, 0, bias=False)

        # Batchnorm
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(256)
        self.bn4 = nn.BatchNorm2d(512)

    def forward(self,x):

        x = F.leaky_relu( self.conv1(x) ,.2)

        x = F.leaky_relu( self.conv2(x) ,.2)
        x = self.bn2(x)

        x = F.leaky_relu( self.conv3(x) ,.2)
        x = self.bn3(x)

        x = F.leaky_relu( self.conv4(x) ,.2)
        x = self.bn4(x)

        x = torch.sigmoid( self.conv5(x) ).view(-1,1)

        return x


In [58]:
# %% Generator class (91x91)

class GeneratorNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolution layers
        self.conv1 = nn.ConvTranspose2d(100,512, 4, 1, 0, bias=False)
        self.conv2 = nn.ConvTranspose2d(512,256, 4, 2, 1, bias=False)
        self.conv3 = nn.ConvTranspose2d(256,128, 4, 2, 1, bias=False)
        self.conv4 = nn.ConvTranspose2d(128, 64, 2, 2, 1, bias=False)
        self.conv5 = nn.ConvTranspose2d( 64,  1, 4, 3, 0, bias=False)

        # Batchnorm
        self.bn1 = nn.BatchNorm2d(512)
        self.bn2 = nn.BatchNorm2d(256)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d( 64)

    def forward(self,x):

        x = F.relu( self.bn1(self.conv1(x)) )
        x = F.relu( self.bn2(self.conv2(x)) )
        x = F.relu( self.bn3(self.conv3(x)) )
        x = F.relu( self.bn4(self.conv4(x)) )
        x = torch.tanh( self.conv5(x) )

        return x


In [73]:
# %% Discriminator class (28x28)

class DiscriminatorNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolution layers adapted for 28x28 input
        # Input: 1x28x28
        self.conv1 = nn.Conv2d(  1, 64, 4, 2, 1, bias=False)
        self.conv2 = nn.Conv2d( 64,128, 4, 2, 1, bias=False)
        self.conv3 = nn.Conv2d(128,256, 3, 1, 0, bias=False)
        self.conv4 = nn.Conv2d(256,512, 5, 1, 0, bias=False)
        self.conv5 = nn.Conv2d(512,  1, 1, 1, 0, bias=False)

        # Batchnorm
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(256)
        self.bn4 = nn.BatchNorm2d(512)

    def forward(self,x):

        x = F.leaky_relu( self.conv1(x) ,.2)

        x = F.leaky_relu( self.conv2(x) ,.2)
        x = self.bn2(x)

        x = F.leaky_relu( self.conv3(x) ,.2)
        x = self.bn3(x)

        x = F.leaky_relu( self.conv4(x) ,.2)
        x = self.bn4(x)

        x = torch.sigmoid( self.conv5(x) ).view(-1,1)

        return x


In [74]:
# %% Generator class (28x28)

class GeneratorNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolution layers adapted for 28x28 output
        # Input: 100x1x1 (noise vector)
        self.conv1 = nn.ConvTranspose2d(100, 256, 4, 1, 0, bias=False)
        self.conv2 = nn.ConvTranspose2d(256, 128, 3, 2, 1, bias=False)
        self.conv3 = nn.ConvTranspose2d(128,  64, 4, 2, 1, bias=False)
        self.conv4 = nn.ConvTranspose2d(64,    1, 4, 2, 1, bias=False)

        # Batchnorm
        self.bn1 = nn.BatchNorm2d(256)
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d( 64)

    def forward(self,x):

        x = F.relu( self.bn1(self.conv1(x)) )
        x = F.relu( self.bn2(self.conv2(x)) )
        x = F.relu( self.bn3(self.conv3(x)) )
        x = torch.tanh( self.conv4(x) )

        return x
